---
title: "3.6.c Riesgo por encharcamientos y vendavales"
short_title: "c. Riesgo"
subtitle: "Depresiones topográficas urbanas, edificaciones expuestas y vulnerabilidad de cubiertas"
description: "Exposición de Leticia a encharcamientos en depresiones topográficas y a vendavales."
keywords: [inundación pluvial, encharcamiento, vendaval, exposición, Leticia]
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/leticia/blob/main/book/03_analisis/36_lluvias_vendavales/c_riesgo.ipynb)

In [ ]:
# Preparación del entorno. En Colab instala el paquete del proyecto desde GitHub (1–2 minutos);
# en una copia local usa el paquete ya instalado con `pip install -e .`.
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "leticia @ git+https://github.com/mromerot/leticia"], check=True)
from leticia.libro import preparar, intentar
preparar()

ThinkHazard califica como **alta** la amenaza por inundación urbana en Leticia [@thinkhazard2024]. En una ciudad
plana y lluviosa, el agua de los aguaceros se acumula en las **depresiones** del terreno y en los fondos de las
quebradas urbanas (San Antonio, Simón Bolívar, Yahuarcaca) cuando el drenaje es insuficiente o está obstruido.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from leticia import aoi, espacial, exposicion, fuentes, riesgo, viz

dem = fuentes.raster_estatico("cop-dem-glo-30", "data")
b = fuentes.bbox("bbox_urbano", 0.01)
dem_u = dem.sel(x=slice(b[0], b[2]), y=slice(b[3], b[1]))
prof = espacial.profundidad_depresiones(dem_u)
edif = exposicion.edificaciones_urbanas()
pts = edif.representative_point().to_crs(4326)
edif["prof_depresion_m"] = np.nan_to_num(prof.interp(x=("p", pts.x.values), y=("p", pts.y.values), method="nearest").values)

## 1. Depresiones topográficas en la ciudad

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 7))
viz.fondo(ax)
v = prof.where(prof >= 0.5)
im = ax.imshow(v.values, extent=(float(prof.x.min()), float(prof.x.max()), float(prof.y.min()), float(prof.y.max())), cmap="Blues",
               vmin=0.5, vmax=5, zorder=2, alpha=0.9)
aoi.area_urbana().to_crs(4326).boundary.plot(ax=ax, color="yellow", lw=0.8, zorder=3)
fig.colorbar(im, ax=ax, shrink=0.6, label="profundidad de la depresión (m)")
viz.marco(ax, titulo="Depresiones topográficas (DEM con sumideros rellenos)")
tabla = pd.DataFrame({u: {"edificaciones": int((edif.prof_depresion_m >= u).sum()), "personas": edif.loc[edif.prof_depresion_m >= u, "poblacion"].sum()}
                      for u in (0.5, 1.0, 2.0)}).T
tabla.index = [f"en depresión ≥ {u} m" for u in tabla.index]
display(tabla.round(0))

In [ ]:
viz.pie(viz.nota_fuente("elevacion", "edificaciones"))

:::{note} Un indicador de tamizaje
El modelo de superficie de 30 m incluye árboles y techos, de modo que algunas «depresiones» son claros entre copas
o patios. El indicador sirve para **priorizar dónde verificar** con la comunidad y con el catastro de redes de
alcantarillado pluvial, no para delimitar zonas inundables.
:::

## 2. Riesgo relativo por encharcamiento

In [ ]:
malla = exposicion.malla_exposicion()
malla["prof_depresion_m"] = espacial.estadistica_zonal(prof.where(prof >= 0.5, 0), malla, "mean")
malla["frac_edif_depresion"] = edif.assign(d=edif.prof_depresion_m >= 0.5).groupby("id_celda")["d"].mean().reindex(malla.index)
amenaza = 0.5 * riesgo.normalizar(malla["prof_depresion_m"].fillna(0)) + 0.5 * riesgo.normalizar(malla["frac_edif_depresion"].fillna(0))
expo = 0.5 * riesgo.normalizar(malla["densidad_construida"]) + 0.5 * riesgo.normalizar(malla["poblacion"])
malla["riesgo_pluvial"] = riesgo.matriz_riesgo(riesgo.clasificar(amenaza), riesgo.clasificar(expo.where(malla.n_edif > 0)))
con = malla[malla.n_edif > 0]
fig, ax = plt.subplots(figsize=(6.4, 7))
viz.mapa_celdas(con, "riesgo_pluvial", ax=ax, categorias=riesgo.CLASES, colores=riesgo.COLORES_CLASES, etiqueta="riesgo relativo",
                titulo="Riesgo relativo por encharcamiento")
display(con.groupby("riesgo_pluvial", observed=False)[["n_edif", "poblacion"]].sum().round(0))
malla[["prof_depresion_m", "frac_edif_depresion", "riesgo_pluvial"]].to_csv(fuentes.ruta("riesgo_pluvial_celdas.csv"))

## 3. Vendavales: un riesgo de toda la ciudad

In [ ]:
ung = fuentes.emergencias_ungrd()
di = fuentes.desinventar()
v = pd.concat([di[di.evento.str.contains("Vendaval", na=False)][["vivafec", "vivdest", "afectados"]].rename(columns={"vivafec": "averiadas", "vivdest": "destruidas"}),
               ung[ung.evento.str.contains("VENDAVAL", na=False)][["viviendas_averiadas", "viviendas_destruidas", "personas"]].rename(
                   columns={"viviendas_averiadas": "averiadas", "viviendas_destruidas": "destruidas", "personas": "afectados"})])
resumen = pd.Series({"eventos registrados": len(v), "viviendas averiadas (total)": v["averiadas"].sum(), "viviendas destruidas (total)": v["destruidas"].sum(),
                     "personas afectadas (total)": v["afectados"].sum(), "edificaciones en la ciudad": len(edif)})
display(resumen.round(0).astype(int).to_frame("valor"))

**Lectura.** Toda edificación de la ciudad está expuesta a un vendaval; la vulnerabilidad depende sobre todo de
la **cubierta** (láminas de zinc o fibrocemento mal ancladas) y del arbolado cercano. El catastro de Leticia no
registra el material de las cubiertas en datos abiertos, así que este libro no puede mapear esa vulnerabilidad.
Programas de anclaje de techos, poda preventiva y sistemas de alerta por tormentas convectivas (IDEAM) son las
medidas de menor costo y mayor efecto.